In [ ]:
import sys
print(sys.executable)

In [3]:
import sys
!{sys.executable} -m pip install sentence-transformers
!{sys.executable} -m pip install faiss-cpu scikit-learn


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\RACHNA\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\RACHNA\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [4]:
import sys
!{sys.executable} -m pip install tf-keras


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\RACHNA\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [5]:
from sentence_transformers import SentenceTransformer

C:\Users\RACHNA\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [6]:
!pip install sentence-transformers
!pip install faiss-cpu
!pip install scikit-learn
!pip install pandas numpy matplotlib

In [7]:
!pip install sentence-transformers faiss-cpu scikit-learn

In [8]:
import numpy as np
import pandas as pd
import re

from sklearn.datasets import fetch_20newsgroups
from sentence_transformers import SentenceTransformer

In [9]:
dataset = fetch_20newsgroups(
    subset="all",
    remove=("headers", "footers", "quotes")
)

documents = dataset.data
labels = dataset.target

print("Total documents:", len(documents))
print("Number of categories:", len(dataset.target_names))

Total documents: 18846
Number of categories: 20


In [10]:
print(documents[0][:500])



I am sure some bashers of Pens fans are pretty confused about the lack
of any kind of posts about the recent Pens massacre of the Devils. Actually,
I am  bit puzzled too and a bit relieved. However, I am going to put an end
to non-PIttsburghers' relief with a bit of praise for the Pens. Man, they
are killing those Devils worse than I thought. Jagr just showed you why
he is much better than his regular season stats. He is also a lot
fo fun to watch in the playoffs. Bowman should let JAgr have a


In [11]:
def clean_text(text):

    text = text.lower()

    # remove email addresses
    text = re.sub(r'\S+@\S+', '', text)

    # remove urls
    text = re.sub(r'http\S+|www\S+', '', text)

    # remove special characters but keep letters, numbers, spaces
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)

    # remove extra spaces
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

In [12]:
clean_documents = [clean_text(doc) for doc in documents]

print(clean_documents[0][:500])

i am sure some bashers of pens fans are pretty confused about the lack of any kind of posts about the recent pens massacre of the devils actually i am bit puzzled too and a bit relieved however i am going to put an end to non pittsburghers relief with a bit of praise for the pens man they are killing those devils worse than i thought jagr just showed you why he is much better than his regular season stats he is also a lot fo fun to watch in the playoffs bowman should let jagr have a lot of fun i


In [13]:
with open("clean_documents.txt", "w", encoding="utf-8") as f:
    for doc in clean_documents:
        f.write(doc.replace("\n", " ") + "\n")

In [14]:
model = SentenceTransformer("all-MiniLM-L6-v2")

In [15]:
embeddings = model.encode(
    clean_documents,
    show_progress_bar=True
)

Batches:   0%|          | 0/589 [00:00<?, ?it/s]

In [16]:
print(embeddings.shape)

(18846, 384)


In [17]:
import faiss

In [18]:
embeddings = embeddings.astype("float32")

In [19]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

In [20]:
index.add(embeddings)

print("Total vectors stored:", index.ntotal)

Total vectors stored: 18846


In [21]:
query = "space exploration and nasa missions"

query_embedding = model.encode([query]).astype("float32")

distances, indices = index.search(query_embedding, k=5)

for i in indices[0]:
    print(clean_documents[i][:300])
    print("\n-----------------\n")

the aiaa san gabriel valley section is sponsoring the following lecture on mars exploration at the jet propulsion lab admission is free and open to the public the next frontier the challenge of mars exploration date may 6 1993 time 6 00pm 8 30 pm location von karman auditorium jet propulsion lab 480

-----------------

i know that alot of how people think and act in a long distance space project would be much like old tiem explorers sailors hunters and such who spent alot of time alone isolated and alone or in minimal surroundings and sopcial contacts such as the old arctic and antarctic expeditions and such i vot

-----------------

lets hear it for dan goldin now if he can only convince the rest of our federal government that the space program is a worth while investment i hope that i will live to see the day we walk on mars but we need to address the technical hurdles first if there s sufficient interest maybe we should consi

-----------------

we ve been progressing towards that g

In [22]:
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture

In [23]:
pca = PCA(n_components=50, random_state=42)

reduced_embeddings = pca.fit_transform(embeddings)

print(reduced_embeddings.shape)

(18846, 50)


In [25]:
bic_scores = []
cluster_range = range(5, 30)

for k in cluster_range:
    gmm = GaussianMixture(n_components=k, random_state=42)
    gmm.fit(reduced_embeddings)
    bic_scores.append(gmm.bic(reduced_embeddings))

In [26]:
best_k = cluster_range[np.argmin(bic_scores)]

print("Best number of clusters:", best_k)

Best number of clusters: 11


In [27]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words="english"
)

tfidf_matrix = vectorizer.fit_transform(clean_documents)

feature_names = vectorizer.get_feature_names_out()

In [28]:
cluster_probabilities = gmm.predict_proba(reduced_embeddings)

print(cluster_probabilities.shape)

(18846, 29)


In [29]:
cluster_labels = cluster_probabilities.argmax(axis=1)

In [30]:
import numpy as np

for cluster in range(best_k):

    print("\nCluster", cluster)

    cluster_docs = np.where(cluster_labels == cluster)[0]

    cluster_tfidf = tfidf_matrix[cluster_docs].mean(axis=0)

    top_words = np.argsort(cluster_tfidf).tolist()[0][-10:]

    for word in reversed(top_words):
        print(feature_names[word])


Cluster 0
drive
scsi
disk
drives
ide
hard
controller
disks
mac
floppy

Cluster 1
space
orbit
shuttle
nasa
launch
moon
earth
like
mission
just

Cluster 2
mail
address
thanks
know
list
send
email
information
like
computer

Cluster 3
government
people
clinton
don
just
like
tax
think
president
drugs

Cluster 4
zone
zip
zionist
zero
zealand
z5
yx
yugoslavia
youth
young

Cluster 5
god
believe
people
religion
don
atheism
say
atheists
think
faith

Cluster 6
windows
dos
file
mouse
program
os
use
files
problem
ms

Cluster 7
just
don
think
like
post
article
know
say
said
ax

Cluster 8
window
server
motif
xterm
application
use
display
widget
x11r5
windows

Cluster 9
bike
car
ride
riding
just
dod
don
dog
road
helmet

Cluster 10
printer
print
fonts
windows
hp
laser
printers
font
printing
driver


In [31]:
confidence = cluster_probabilities.max(axis=1)

In [32]:
boundary_docs = np.argsort(confidence)[:10]

In [33]:
for i in boundary_docs:
    
    print("\nDocument ID:", i)
    print("Cluster probabilities:", cluster_probabilities[i])
    print("\nText:")
    print(clean_documents[i][:400])
    print("\n-----------------------------\n")


Document ID: 6796
Cluster probabilities: [7.70627025e-07 3.75957228e-25 1.46646265e-10 1.26116862e-44
 0.00000000e+00 0.00000000e+00 1.11930139e-01 6.09668735e-25
 4.46375459e-01 4.91305556e-30 2.80416662e-05 4.41552162e-01
 0.00000000e+00 7.61469135e-14 0.00000000e+00 0.00000000e+00
 0.00000000e+00 1.18917918e-23 4.98547382e-39 2.81548528e-05
 1.08238661e-11 0.00000000e+00 1.02714674e-19 0.00000000e+00
 0.00000000e+00 0.00000000e+00 9.03042132e-39 0.00000000e+00
 8.29457640e-05]

Text:
it isn t use tar on the un x box and gtak110 zip on the dos box needs aspi driver ralf

-----------------------------


Document ID: 2652
Cluster probabilities: [9.32976918e-25 2.18094154e-11 4.54266995e-01 5.83695825e-15
 0.00000000e+00 0.00000000e+00 2.39870301e-21 2.96785838e-12
 4.13960298e-32 4.62146431e-01 5.77728832e-27 9.80577218e-21
 0.00000000e+00 1.94453065e-10 1.27062193e-36 2.04012819e-30
 4.05843012e-35 9.92254820e-03 4.86289910e-22 1.03058424e-20
 7.36647695e-02 2.64988369e-19 1.87103783

In [34]:
semantic_cache = {
    cluster: {
        "queries": [],
        "embeddings": [],
        "results": []
    }
    for cluster in range(best_k)
}

In [35]:
from sklearn.metrics.pairwise import cosine_similarity

In [36]:
def search_cache(query_embedding, cluster):

    cache = semantic_cache[cluster]

    if len(cache["embeddings"]) == 0:
        return None

    similarities = cosine_similarity(
        query_embedding,
        cache["embeddings"]
    )[0]

    best_match = similarities.argmax()
    best_score = similarities[best_match]

    if best_score > SIMILARITY_THRESHOLD:
        return cache["results"][best_match]

    return None

In [37]:
def add_to_cache(query, query_embedding, results):

    semantic_cache["queries"].append(query)

    semantic_cache["embeddings"].append(query_embedding[0])

    semantic_cache["results"].append(results)

In [38]:
def semantic_search(query):

    query_embedding = model.encode([query]).astype("float32")

    cluster = get_query_cluster(query_embedding)

    cached_result = search_cache(query_embedding, cluster)

    if cached_result is not None:
        print("CACHE HIT")
        return cached_result

    print("CACHE MISS")

    distances, indices = index.search(query_embedding, k=5)

    results = [clean_documents[i] for i in indices[0]]

    semantic_cache[cluster]["queries"].append(query)
    semantic_cache[cluster]["embeddings"].append(query_embedding[0])
    semantic_cache[cluster]["results"].append(results)

    return results

In [40]:
def get_query_cluster(query_embedding):
    
    query_reduced = pca.transform(query_embedding)
    
    probs = gmm.predict_proba(query_reduced)
    
    cluster = probs.argmax(axis=1)[0]
    
    return cluster

In [58]:
SIMILARITY_THRESHOLD = 0.6

In [59]:
result1 = semantic_search("mars exploration missions")

result2 = semantic_search("nasa missions to mars")

CACHE MISS
CACHE HIT (similarity=0.900)


In [60]:
semantic_cache = {
    cluster: {
        "queries": [],
        "embeddings": [],
        "results": []
    }
    for cluster in range(best_k)
}

In [61]:
from sklearn.metrics.pairwise import cosine_similarity

In [62]:
def search_cache(query_embedding, cluster):

    cache = semantic_cache.setdefault(cluster, {
    "queries": [],
    "embeddings": [],
    "results": []
})

    if len(cache["embeddings"]) == 0:
        return None

    similarities = cosine_similarity(
        query_embedding,
        cache["embeddings"]
    )[0]

    best_match = similarities.argmax()
    best_score = similarities[best_match]

    if best_score > SIMILARITY_THRESHOLD:
        print(f"CACHE HIT (similarity={best_score:.3f})")
        return cache["results"][best_match]

    return None

In [63]:
def add_to_cache(query, query_embedding, results, cluster):

    semantic_cache[cluster]["queries"].append(query)
    semantic_cache[cluster]["embeddings"].append(query_embedding[0])
    semantic_cache[cluster]["results"].append(results)

In [64]:
def semantic_search(query):

    query_embedding = model.encode([query]).astype("float32")

    cluster = get_query_cluster(query_embedding)

    cached_result = search_cache(query_embedding, cluster)

    if cached_result is not None:
        return cached_result

    print("CACHE MISS")

    distances, indices = index.search(query_embedding, k=5)

    results = [clean_documents[i] for i in indices[0]]

    add_to_cache(query, query_embedding, results, cluster)

    return results

In [65]:
r1 = semantic_search("mars exploration missions")

r2 = semantic_search("nasa missions to mars")

r3 = semantic_search("hockey team players")

print(r3[0][:300])

CACHE MISS
CACHE HIT (similarity=0.900)
CACHE MISS
this is an all point team for the canadian nhlers who are not playoff bound goalies bill ranford edmonton sean burke hartford peter sidorkiewicz ottawa defensemen zarley zalapski hartford norm maciver ottawa garry galley philadelphia greg hawgood philadelphia dave manson edmonton mark tinordi minnes


In [66]:
pip install fastapi uvicorn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\RACHNA\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [67]:
# imports
from fastapi import FastAPI
from pydantic import BaseModel
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# load model
model = SentenceTransformer("all-MiniLM-L6-v2")

# load dataset
with open("clean_documents.txt", "r", encoding="utf-8") as f:
    clean_documents = [line.strip() for line in f]

# create embeddings
embeddings = model.encode(clean_documents).astype("float32")

# build FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

# clustering setup
best_k = 10

semantic_cache = {
    cluster: {
        "queries": [],
        "embeddings": [],
        "results": []
    }
    for cluster in range(best_k)
}


# ------------------------------------------------
# FASTAPI APP
# ------------------------------------------------

app = FastAPI()


# stats
total_entries = 0
hit_count = 0
miss_count = 0


class QueryRequest(BaseModel):
    query: str


# ------------------------------------------------
# API ROUTES
# ------------------------------------------------

@app.post("/query")
def query_endpoint(request: QueryRequest):

    global hit_count, miss_count, total_entries

    query = request.query

    query_embedding = model.encode([query]).astype("float32")

    cluster = get_query_cluster(query_embedding)

    cached_result = search_cache(query_embedding, cluster)

    if cached_result is not None:
        hit_count += 1

        return {
            "query": query,
            "cache_hit": True,
            "result": cached_result,
            "dominant_cluster": int(cluster)
        }

    miss_count += 1

    distances, indices = index.search(query_embedding, k=5)

    results = [clean_documents[i] for i in indices[0]]

    add_to_cache(query, query_embedding, results, cluster)

    total_entries = sum(len(semantic_cache[c]["queries"]) for c in semantic_cache)

    return {
        "query": query,
        "cache_hit": False,
        "result": results,
        "dominant_cluster": int(cluster)
    }


@app.get("/cache/stats")
def cache_stats():

    total = hit_count + miss_count
    hit_rate = (hit_count / total) * 100 if total > 0 else 0

    return {
        "total_entries": total_entries,
        "hit_count": hit_count,
        "miss_count": miss_count,
        "hit_rate": f"{hit_rate:.2f}%"
    }


@app.delete("/cache")
def clear_cache():

    global semantic_cache, hit_count, miss_count, total_entries

    semantic_cache = {
        cluster: {
            "queries": [],
            "embeddings": [],
            "results": []
        }
        for cluster in range(best_k)
    }

    hit_count = 0
    miss_count = 0
    total_entries = 0

    return {"message": "Cache cleared"}